# 09 — Ensemble Comparison: Multiple Engines on the Same AOI

Runs **Delineate-Anything**, **FTW**, and **GeoAI** on the same study area (Andalusia, Spain), then runs the **ensemble** engine with a vote-based merge strategy. Compares per-engine and consensus results.

**Estimated runtime:** ~30–60 minutes (runs 2–3 engines, GPU recommended).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything,ftw,geoai]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/ensemble_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "sentinel2"
YEAR = 2024
ENGINES = ["delineate-anything", "ftw", "geoai"]

## Create Study Area

AOI in southern Spain (Andalusia) — a mix of field sizes.

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-3.80, 37.75],
                        [-3.65, 37.75],
                        [-3.65, 37.85],
                        [-3.80, 37.85],
                        [-3.80, 37.75],
                    ]
                ],
            },
            "properties": {"name": "Andalusia AOI"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "andalusia_aoi.geojson")
with open(study_area, "w") as f:
    json.dump(aoi, f)

print(f"Study area: {study_area}")

## Run Each Engine Individually

In [ ]:
results = {}

for engine_name in ENGINES:
    print(f"Running engine: {engine_name}...", end=" ")
    output_path = OUTPUT_DIR / f"fields_{engine_name}.gpkg"

    try:
        gdf = agribound.delineate(
            study_area=study_area,
            source=SOURCE,
            year=YEAR,
            engine=engine_name,
            output_path=str(output_path),
            gee_project=GEE_PROJECT,
            min_area=2500,
            simplify=2.0,
        )
        results[engine_name] = gdf
        print(f"{len(gdf)} fields")
    except Exception as exc:
        print(f"FAILED: {exc}")

## Run Ensemble Engine (Vote Strategy)

In [ ]:
try:
    ensemble_gdf = agribound.delineate(
        study_area=study_area,
        source=SOURCE,
        year=YEAR,
        engine="ensemble",
        output_path=str(OUTPUT_DIR / "fields_ensemble.gpkg"),
        gee_project=GEE_PROJECT,
        engine_params={
            "engines": ENGINES,
            "merge_strategy": "vote",
        },
        min_area=2500,
    )
    results["ensemble"] = ensemble_gdf
    print(f"Ensemble: {len(ensemble_gdf)} fields")
except Exception as exc:
    print(f"Ensemble failed: {exc}")

## Comparison Summary

In [ ]:
for name, gdf in results.items():
    n = len(gdf)
    avg_area = gdf["metrics:area"].mean() / 10000 if "metrics:area" in gdf.columns else 0
    print(f"{name:<25} {n:>5} fields, avg {avg_area:>6.1f} ha")

## Visualization

In [ ]:
if len(results) >= 2:
    from agribound.visualize import show_comparison

    m = show_comparison(
        list(results.values()),
        labels=list(results.keys()),
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_comparison.html"),
    )
    m